In [2]:
# install dependencies 
install.packages(c(
  'tidyverse',
  'tidymodels',
  'janitor',
  'skimr',
  'glmnet',
  'ranger',
  'pROC',
  'rlang'
))

# load the libraries
library(tidyverse)
library(tidymodels)
library(janitor)
library(skimr)
library(glmnet)
library(ranger)
library(pROC)
library(rlang)


trying URL 'https://cran.rstudio.com/bin/macosx/big-sur-arm64/contrib/4.5/tidyverse_2.0.0.tgz'
trying URL 'https://cran.rstudio.com/bin/macosx/big-sur-arm64/contrib/4.5/tidymodels_1.5.0.tgz'
trying URL 'https://cran.rstudio.com/bin/macosx/big-sur-arm64/contrib/4.5/janitor_2.2.1.tgz'
trying URL 'https://cran.rstudio.com/bin/macosx/big-sur-arm64/contrib/4.5/skimr_2.2.2.tgz'
trying URL 'https://cran.rstudio.com/bin/macosx/big-sur-arm64/contrib/4.5/glmnet_5.0.tgz'
trying URL 'https://cran.rstudio.com/bin/macosx/big-sur-arm64/contrib/4.5/ranger_0.18.0.tgz'
trying URL 'https://cran.rstudio.com/bin/macosx/big-sur-arm64/contrib/4.5/pROC_1.19.0.1.tgz'
trying URL 'https://cran.rstudio.com/bin/macosx/big-sur-arm64/contrib/4.5/rlang_1.2.0.tgz'



The downloaded binary packages are in
	/var/folders/rg/flyq5j697ysbkxyry2m15zr80000gn/T//RtmpaYyUzR/downloaded_packages

Attaching package: ‘rlang’

The following objects are masked from ‘package:purrr’:

    flatten, flatten_chr, flatten_dbl, flatten_int, flatten_lgl,
    flatten_raw, invoke, splice



In [3]:
# load dataset file paths
path_2023 <- 'flight_with_weather_2023.csv'
path_2024 <- 'flight_with_weather_2024.csv'

In [4]:
# read the data
flight_2023 <- read_csv(path_2023)
flight_2024 <- read_csv(path_2024)

Rows: 6645461 Columns: 34
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (3): OP_CARRIER, ORIGIN, DEST
dbl  (24): OP_CARRIER_FL_NUM, DEP_DELAY, TAXI_OUT, TAXI_IN, ARR_DELAY, CRS_E...
dttm  (7): FL_DATE, CRS_DEP_TIME, DEP_TIME, WHEELS_OFF, WHEELS_ON, CRS_ARR_T...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 6284841 Columns: 34
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (3): OP_CARRIER, ORIGIN, DEST
dbl  (24): OP_CARRIER_FL_NUM, DEP_DELAY, TAXI_OUT, TAXI_IN, ARR_DELAY, CRS_E...
dttm  (7): FL_DATE, CRS_DEP_TIME, DEP_TIME, WHEELS_OFF, WHEELS_ON, CRS_ARR_T...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
# inspect column names 2023
names(flight_2023)

 [1] "FL_DATE"             "OP_CARRIER"          "OP_CARRIER_FL_NUM"  
 [4] "ORIGIN"              "DEST"                "CRS_DEP_TIME"       
 [7] "DEP_TIME"            "DEP_DELAY"           "TAXI_OUT"           
[10] "WHEELS_OFF"          "WHEELS_ON"           "TAXI_IN"            
[13] "CRS_ARR_TIME"        "ARR_TIME"            "ARR_DELAY"          
[16] "CRS_ELAPSED_TIME"    "ACTUAL_ELAPSED_TIME" "AIR_TIME"           
[19] "FLIGHTS"             "MONTH"               "DAY_OF_MONTH"       
[22] "DAY_OF_WEEK"         "ORIGIN_INDEX"        "DEST_INDEX"         
[25] "O_TEMP"              "O_PRCP"              "O_WSPD"             
[28] "D_TEMP"              "D_PRCP"              "D_WSPD"             
[31] "O_LATITUDE"          "O_LONGITUDE"         "D_LATITUDE"         
[34] "D_LONGITUDE"        

In [6]:
# inspect column names 2024
names(flight_2024)

 [1] "FL_DATE"             "OP_CARRIER"          "OP_CARRIER_FL_NUM"  
 [4] "ORIGIN"              "DEST"                "CRS_DEP_TIME"       
 [7] "DEP_TIME"            "DEP_DELAY"           "TAXI_OUT"           
[10] "WHEELS_OFF"          "WHEELS_ON"           "TAXI_IN"            
[13] "CRS_ARR_TIME"        "ARR_TIME"            "ARR_DELAY"          
[16] "CRS_ELAPSED_TIME"    "ACTUAL_ELAPSED_TIME" "AIR_TIME"           
[19] "FLIGHTS"             "MONTH"               "DAY_OF_MONTH"       
[22] "DAY_OF_WEEK"         "ORIGIN_INDEX"        "DEST_INDEX"         
[25] "O_TEMP"              "O_PRCP"              "O_WSPD"             
[28] "D_TEMP"              "D_PRCP"              "D_WSPD"             
[31] "O_LATITUDE"          "O_LONGITUDE"         "D_LATITUDE"         
[34] "D_LONGITUDE"        

In [7]:
# verify both datasets have identical column names and order
identical(names(flight_2023), names(flight_2024))

[1] TRUE

In [8]:
# check unique values for MONTH column 2023
unique(flight_2023$MONTH)

 [1]  1  2  3  4  5  6  7  8  9 10 11 12

In [9]:
# check unique values for MONTH column 2024
unique(flight_2024$MONTH)

 [1]  1  2  4  5  6  7  8  9 10 11 12

In [10]:
# filter to only keep Dec rows 
dec_flight_2023 <- flight_2023 %>%
  filter(MONTH == 12) %>%
  mutate(YEAR = 2023L)

dec_flight_2024 <- flight_2024 %>%
  filter(MONTH == 12) %>%
  mutate(YEAR = 2024L)

In [ ]:
# confirm only December rows remain 2023
unique(dec_flight_2023$MONTH)

[1] 12

In [12]:
# confirm only December rows remain 2024
unique(dec_flight_2024$MONTH)

[1] 12

In [17]:
cat("December 2023:", nrow(dec_flight_2023), "flights\n")
cat("December 2024:", nrow(dec_flight_2024), "flights\n")

December 2023: 558609 flights
December 2024: 574434 flights


In [18]:
# combined both dataset and rename columns to snake_case
df <- bind_rows(dec_flight_2023, dec_flight_2024) %>%
  janitor::clean_names()

In [19]:
glimpse(df)

Rows: 1,133,043
Columns: 35
$ fl_date             <dttm> 2023-12-01, 2023-12-01, 2023-12-01, 2023-12-02, 2…
$ op_carrier          <chr> "DL", "OO", "OO", "DL", "OO", "OO", "DL", "OO", "O…
$ op_carrier_fl_num   <dbl> 1189, 3888, 3920, 1189, 3888, 3920, 1189, 3888, 39…
$ origin              <chr> "DLH", "DLH", "DLH", "DLH", "DLH", "DLH", "DLH", "…
$ dest                <chr> "MSP", "MSP", "MSP", "MSP", "MSP", "MSP", "MSP", "…
$ crs_dep_time        <dttm> 2023-12-01 05:05:00, 2023-12-01 10:44:00, 2023-12…
$ dep_time            <dttm> 2023-12-01 05:23:00, 2023-12-01 10:36:00, 2023-12…
$ dep_delay           <dbl> 18, -8, -5, -7, 0, -5, 1, -6, -8, -8, -7, -5, 0, -…
$ taxi_out            <dbl> 20, 12, 21, 23, 14, 25, 16, 29, 21, 11, 10, 35, 26…
$ wheels_off          <dttm> 2023-12-01 05:43:00, 2023-12-01 10:48:00, 2023-12…
$ wheels_on           <dttm> 2023-12-01 06:14:00, 2023-12-01 11:20:00, 2023-12…
$ taxi_in             <dbl> 3, 4, 5, 6, 7, 4, 4, 5, 5, 3, 3, 3, 5, 5, 8, 3, 4,…
$ crs_arr_ti

In [20]:
skim(df)

── Data Summary ────────────────────────
                           Values 
Name                       df     
Number of rows             1133043
Number of columns          35     
_______________________           
Column type frequency:            
  character                3      
  numeric                  25     
  POSIXct                  7      
________________________          
Group variables            None   

── Variable type: character ────────────────────────────────────────────────────
  skim_variable n_missing complete_rate min max empty n_unique whitespace
1 op_carrier            0             1   2   2     0       15          0
2 origin                0             1   3   3     0      298          0
3 dest                  0             1   3   3     0      298          0

── Variable type: numeric ──────────────────────────────────────────────────────
   skim_variable       n_missing complete_rate     mean       sd     p0    p25
 1 op_carrier_fl_num           0   

In [24]:
# summary statistics by year
df %>%
  group_by(year) %>%
  summarise(
    n_flights        = n(),
    mean_arr_delay   = round(mean(arr_delay, na.rm = TRUE), 2),
    median_arr_delay = round(median(arr_delay, na.rm = TRUE), 2),
    delay_rate_pct   = round(mean(arr_delay > 15, na.rm = TRUE) * 100, 2),
    mean_dep_delay   = round(mean(dep_delay, na.rm = TRUE), 2),
    n_carriers       = n_distinct(op_carrier),
    n_airports       = n_distinct(origin),
    .groups = "drop"
  )

# A tibble: 2 × 8
   year n_flights mean_arr_delay median_arr_delay delay_rate_pct mean_dep_delay
  <int>     <int>          <dbl>            <dbl>          <dbl>          <dbl>
1  2023    558609           0.43               -8           15.0           8.16
2  2024    574434           6.95               -6           20.4          13.3 
# ℹ 2 more variables: n_carriers <int>, n_airports <int>